# 🔋 Semantic Energy — Live Deployment

This notebook deploys the **Semantic Energy** app on Google Colab with a public URL via **ngrok**.

**What you get:**
- Full FastAPI backend with Qwen 2.5 1.5B (or Llama 3 8B with 4-bit quantization)
- Chat frontend served at a public URL
- Full per-token logit access for uncertainty estimation

---

## Prerequisites

1. **Runtime → Change runtime type → T4 GPU**
2. Get a **free ngrok auth token** from [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)

> ⚠️ **Important:** Make sure you've selected a **T4 GPU runtime** before running the cells below.

## Step 1: Verify GPU

In [ ]:
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)")
else:
    print("❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

## Step 2: Install Dependencies

In [ ]:
!pip install -q transformers accelerate fastapi uvicorn pyngrok
print("✅ Dependencies installed!")

## Step 3: Clone the Repository

In [ ]:
import os

REPO_URL = "https://github.com/DangerDudeSL/SemanticEnergyV1.git"
REPO_DIR = "SemanticEnergyV1"

if os.path.exists(REPO_DIR):
    print(f"📁 {REPO_DIR} already exists, pulling latest...")
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL}
    print(f"✅ Cloned {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"📂 Working directory: {os.getcwd()}")

## Step 4: Set Up ngrok

Get your **free auth token** from [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken) and paste it below.

In [ ]:
NGROK_AUTH_TOKEN = ""  # ← Paste your ngrok auth token here

if not NGROK_AUTH_TOKEN:
    # If not hardcoded, prompt for it
    NGROK_AUTH_TOKEN = input("Enter your ngrok auth token: ")

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("✅ ngrok authenticated!")

## Step 5: Choose Model

Pick which model to load. Qwen 2.5 1.5B is the default (fast, lightweight). Llama 3 8B gives better answers but requires 4-bit quantization and a Hugging Face token.

In [ ]:
# ===== CONFIGURATION =====

# Option 1: Qwen 2.5 1.5B (default — fast, no special access needed)
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
USE_4BIT = False

# Option 2: Llama 3.1 8B (better quality, needs HF token + 4-bit)
# MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
# USE_4BIT = True

# If using a gated model (Llama), uncomment and set your HF token:
# import os
# os.environ["HF_TOKEN"] = "hf_your_token_here"

print(f"📦 Model: {MODEL_ID}")
print(f"🔧 4-bit quantization: {'Yes' if USE_4BIT else 'No'}")

## Step 6: Load the Model & Start the Server

This cell loads the model into GPU memory and starts the FastAPI server with an ngrok public URL.

> ⏳ First run takes **2-5 minutes** (downloading model weights). Subsequent runs are faster due to caching.

In [ ]:
import sys
import os
import threading
import time
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from statistics import mean
from math import prod
import math

# ---- Load the math helpers from engine.py ----
sys.path.insert(0, os.path.join(os.getcwd(), "backend"))
from engine import SemanticEngine, cal_flow, sum_normalize

# ---- Optionally override engine to use 4-bit quantization ----
print(f"\n🚀 Loading model: {MODEL_ID}", flush=True)

if USE_4BIT:
    !pip install -q bitsandbytes
    from transformers import BitsAndBytesConfig
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        device_map="auto",
    )
    # Monkey-patch a SemanticEngine instance with the quantized model
    engine = SemanticEngine.__new__(SemanticEngine)
    engine.device = model.device
    engine.tokenizer = tokenizer
    engine.model = model
else:
    engine = SemanticEngine(model_id=MODEL_ID)

print(f"✅ Model loaded on {engine.device}!")
if torch.cuda.is_available():
    mem_used = torch.cuda.memory_allocated() / 1e9
    print(f"📊 GPU memory used: {mem_used:.1f} GB")

In [ ]:
import traceback
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, FileResponse, HTMLResponse
from fastapi.staticfiles import StaticFiles
import uvicorn

app = FastAPI(title="Semantic Energy API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Serve the frontend
frontend_dir = os.path.join(os.getcwd(), "frontend")

@app.get("/")
async def serve_index():
    return FileResponse(os.path.join(frontend_dir, "index.html"))

@app.get("/styles.css")
async def serve_css():
    return FileResponse(os.path.join(frontend_dir, "styles.css"), media_type="text/css")

@app.get("/script.js")
async def serve_js():
    return FileResponse(os.path.join(frontend_dir, "script.js"), media_type="application/javascript")

@app.post("/chat")
async def chat_endpoint(request: Request):
    try:
        data = await request.json()
        user_prompt = data.get("prompt", "")
        num_samples = data.get("num_samples", 5)

        if not user_prompt:
            return JSONResponse({"error": "Prompt is required"}, status_code=400)

        print(f"\n[1/3] Generating {num_samples} responses for: {user_prompt}", flush=True)
        generated_data = engine.generate_responses(user_prompt, num_samples=num_samples)

        main_answer = generated_data[0]["answer"]

        print(f"[2/3] Deciding semantic equivalent clusters...", flush=True)
        answer_texts = [d["answer"] for d in generated_data]
        clusters = engine.find_semantic_clusters(user_prompt, answer_texts)
        print(f"Found {len(clusters)} clusters: {clusters}", flush=True)

        print(f"[3/3] Calculating Semantic Energy / Uncertainty...", flush=True)
        probs_list = [d['probs'] for d in generated_data]
        logits_list = [d['logits'] for d in generated_data]

        probs_se, logits_se = cal_flow(probs_list, logits_list, clusters, fermi_mu=None)

        main_cluster_idx = 0
        for idx, cluster in enumerate(clusters):
            if 0 in cluster:
                main_cluster_idx = idx
                break

        cluster_energies = sum_normalize(logits_se)
        main_confidence = cluster_energies[main_cluster_idx]

        if main_confidence > 0.80:
            confidence_level = "high"
        elif main_confidence > 0.50:
            confidence_level = "medium"
        else:
            confidence_level = "low"

        print(f"✅ Confidence: {main_confidence:.2f} ({confidence_level})", flush=True)

        return {
            "answer": main_answer,
            "confidence_score": main_confidence,
            "confidence_level": confidence_level,
            "clusters_found": len(clusters),
            "debug_data": {
                "all_answers": answer_texts,
                "energies_per_cluster": cluster_energies
            }
        }
    except Exception as e:
        print(f"❌ ERROR: {e}", flush=True)
        traceback.print_exc()
        return JSONResponse({"error": str(e)}, status_code=500)


# --- Start Server + ngrok ---
public_url = ngrok.connect(7860)
print(f"\n{'='*60}")
print(f"🌐 PUBLIC URL: {public_url}")
print(f"{'='*60}")
print(f"\n📋 Share this URL with anyone to let them use the app!")
print(f"💡 The URL stays active as long as this Colab session is running.\n")

# Run uvicorn in a thread so the cell doesn't block
threading.Thread(
    target=uvicorn.run,
    args=(app,),
    kwargs={"host": "0.0.0.0", "port": 7860, "log_level": "warning"},
    daemon=True
).start()

print("🟢 Server is running! Open the URL above in your browser.")
print("⏹️  To stop: Runtime → Disconnect and delete runtime")

## 🧪 Quick Test

Run this cell to verify the API is working and logits are accessible:

In [ ]:
import requests
import json

test_url = "http://localhost:7860/chat"
test_data = {"prompt": "What is the capital of France?", "num_samples": 3}

print("🧪 Testing API...\n")
response = requests.post(test_url, json=test_data)

if response.status_code == 200:
    result = response.json()
    print(f"✅ Answer: {result['answer']}")
    print(f"📊 Confidence: {result['confidence_score']:.2%} ({result['confidence_level']})")
    print(f"🔗 Semantic Clusters: {result['clusters_found']}")
    print(f"\n📈 Cluster Energies: {result['debug_data']['energies_per_cluster']}")
else:
    print(f"❌ Error {response.status_code}: {response.text}")

---

## 📌 Notes

- **Session duration:** Free Colab sessions last ~12 hours max. The ngrok URL changes each time you restart.
- **Model caching:** After the first download, the model is cached in Colab's `/root/.cache/huggingface` until the runtime resets.
- **GPU memory:** Check usage with `!nvidia-smi` in a new cell.
- **Switching models:** Restart the runtime, change the config in Step 5, and re-run all cells.

---

*Powered by [Semantic Energy](https://arxiv.org/abs/2508.14496) — Detecting LLM Hallucination Beyond Entropy*